# Paired Switching: Local Data Research Notebook

This notebook tests and validates the `PairedSwitching` algorithm using **local data** from your data directory.

**Workflow:**
1. **Data Loading:** Load historical price data from local zip files
2. **Universe Selection:** Select top liquid stocks from available data
3. **Metric Collection:** Calculate momentum, volatility, and other metrics
4. **Clustering:** Classify stocks into 6 market regimes
5. **Visualization:** Plot clusters and analyze results

**Data Location:** `/Lean/Data/equity/usa/daily/` (mounted from `C:\Users\kenbr\QC\data` on host)

In [2]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import zipfile
import io
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data configuration
DATA_ROOT = Path("/Lean/Data")
EQUITY_DAILY = DATA_ROOT / "equity" / "usa" / "daily"
SYMBOL_PROPS = DATA_ROOT / "symbol-properties" / "symbol-properties-database.csv"

print(f"[OK] Data root: {DATA_ROOT}")
print(f"[OK] Equity daily data: {EQUITY_DAILY}")
print(f"[OK] Data folder exists: {EQUITY_DAILY.exists()}")
if EQUITY_DAILY.exists():
    zip_count = len(list(EQUITY_DAILY.glob("*.zip")))
    print(f"[OK] Found {zip_count} symbol data files")

[OK] Data root: /Lean/Data
[OK] Equity daily data: /Lean/Data/equity/usa/daily
[OK] Data folder exists: True
[OK] Found 560 symbol data files


In [ ]:
# Cell 2: Helper Functions for Local Data Loading

def load_symbol_properties():
    """Load symbol properties database to get market info."""
    try:
        if SYMBOL_PROPS.exists():
            props_df = pd.read_csv(SYMBOL_PROPS)
            print(f"[OK] Loaded {len(props_df)} symbols from properties database")
            return props_df
        else:
            print("[!] Symbol properties file not found")
            return None
    except Exception as e:
        print(f"[!] Error loading symbol properties: {e}")
        return None


def get_available_symbols(limit=None):
    """Get list of available symbols from local data directory."""
    if not EQUITY_DAILY.exists():
        print(f"[X] Directory not found: {EQUITY_DAILY}")
        return []
    
    zip_files = sorted(EQUITY_DAILY.glob("*.zip"))
    symbols = [f.stem.upper() for f in zip_files]
    
    if limit:
        symbols = symbols[:limit]
    
    print(f"[OK] Found {len(symbols)} available symbols")
    return symbols


def load_symbol_data(symbol, start_date=None, end_date=None):
    """Load historical price data for a single symbol from local zip file."""
    zip_path = EQUITY_DAILY / f"{symbol.lower()}.zip"
    
    if not zip_path.exists():
        return None
    
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            # Usually the csv file inside has the same name as the zip
            csv_name = f"{symbol.lower()}.csv"
            
            # Try to find the CSV file (might have different naming)
            file_list = zf.namelist()
            if csv_name not in file_list:
                csv_name = file_list[0]  # Take first file if exact match not found
            
            with zf.open(csv_name) as f:
                df = pd.read_csv(f)
                
                # Parse date column (typically first column or named 'date'/'time')
                if 'date' in df.columns:
                    df['date'] = pd.to_datetime(df['date'])
                elif 'time' in df.columns:
                    df['date'] = pd.to_datetime(df['time'])
                else:
                    # First column is usually the date
                    df['date'] = pd.to_datetime(df.iloc[:, 0])
                
                df.set_index('date', inplace=True)
                
                # Standardize column names to lowercase
                df.columns = [c.lower() for c in df.columns]
                
                # Filter by date range if provided
                if start_date:
                    df = df[df.index >= start_date]
                if end_date:
                    df = df[df.index <= end_date]
                
                return df
                
    except Exception as e:
        print(f"[!] Error loading {symbol}: {e}")
        return None


def load_multiple_symbols(symbols, start_date, end_date):
    """Load historical data for multiple symbols and combine into single DataFrame."""
    all_data = {}
    
    for symbol in symbols:
        df = load_symbol_data(symbol, start_date, end_date)
        if df is not None and not df.empty:
            all_data[symbol] = df
    
    print(f"[OK] Successfully loaded {len(all_data)}/{len(symbols)} symbols")
    return all_data


                "print(\"[OK] Helper functions defined\")",
                ""
            ]
        },
        {
            "cell_type": "markdown",
            "language": "markdown",
            "source": [
                "## Pre-Processed Metrics Loading",
                "",
                "**Note:** Metrics are now processed independently using `metrics_processor.py`",
                "",
                "To run the processor and generate new metrics:"
            ]
        },
        {
            "cell_type": "code",
            "language": "python",
            "source": [
                "# Cell 3a: Load Pre-Processed Metrics (REPLACES cells 4-5)",
                "# This loads metrics that were computed by metrics_processor.py",
                "",
                "def load_latest_metrics(metrics_dir=\"processed_metrics\"):",
                "    \"\"\"Load the latest metrics CSV from processed_metrics directory.\"\"\"",
                "    metrics_path = Path(metrics_dir)",
                "    ",
                "    if not metrics_path.exists():",
                "        print(f\"[!] Metrics directory not found: {metrics_path.absolute()}\")",
                "        print(f\"[!] Run: python metrics_processor.py\")",
                "        return None",
                "    ",
                "    # Find all metric files",
                "    metric_files = sorted(metrics_path.glob('metrics_*.csv'))",
                "    ",
                "    if not metric_files:",
                "        print(f\"[!] No metric files found in {metrics_path.absolute()}\")",
                "        print(f\"[!] Run: python metrics_processor.py\")",
                "        return None",
                "    ",
                "    # Load latest",
                "    latest_file = metric_files[-1]",
                "    print(f\"[OK] Loading metrics from: {latest_file.name}\")",
                "    ",
                "    df = pd.read_csv(latest_file, index_col=0)",
                "    print(f\"[OK] Loaded metrics for {len(df)} symbols\")",
                "    ",
                "    return df",
                "",
                "",
                "# Load metrics",
                "metrics_df = load_latest_metrics()",
                "",
                "if metrics_df is not None:",
                "    print(f\"\\n[OK] Metrics loaded successfully!\")",
                "    print(f\"   Symbols: {len(metrics_df)}\")",
                "    print(f\"   Columns: {list(metrics_df.columns)[:5]}...\") ",
                "    print(f\"\\n   Sample metrics:\")",
                "    print(metrics_df.head())",
                "else:",
                "    print(\"[X] Failed to load metrics\")"

[OK] Helper functions defined


In [ ]:
# Cell 3: Configuration and Universe Selection

import random

# Analysis date range
END_DATE = datetime(2025, 10, 18)  # Target analysis date
START_DATE = END_DATE - timedelta(days=3*365)  # 3 years of history

print(f"Data range: {START_DATE.date()} to {END_DATE.date()}")

# Load symbol properties to filter further
props_df = load_symbol_properties()

# Get all available symbols from data directory
all_symbols = get_available_symbols()

# For testing: randomly select 100 symbols for faster analysis
random.seed(42)  # For reproducibility
universe_symbols = random.sample(all_symbols, min(10, len(all_symbols)))

print(f"[OK] Universe: {len(universe_symbols)} symbols selected for analysis (from {len(all_symbols)} available)")
print(f"Sample: {universe_symbols[:10]}")

Data range: 2022-10-19 to 2025-10-18
[OK] Loaded 10382 symbols from properties database
[OK] Found 560 available symbols
[OK] Universe: 99 symbols selected for analysis
Sample: ['SPY', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'V', 'JPM']


In [4]:
# Cell 4: Load Historical Price Data

print(f"Loading historical data for {len(universe_symbols)} symbols...")
print(f"   Date range: {START_DATE.date()} to {END_DATE.date()}")

# Load all symbol data
symbol_data = load_multiple_symbols(universe_symbols, START_DATE, END_DATE)

# Display summary
if symbol_data:
    print(f"\n[OK] Data loaded successfully!")
    print(f"   Symbols with data: {len(symbol_data)}")
    
    # Show sample data from first symbol
    sample_symbol = list(symbol_data.keys())[0]
    sample_df = symbol_data[sample_symbol]
    print(f"\n Sample data from {sample_symbol}:")
    print(f"   Shape: {sample_df.shape}")
    print(f"   Columns: {list(sample_df.columns)}")
    print(f"   Date range: {sample_df.index.min()} to {sample_df.index.max()}")
    print(f"\n   First few rows:")
    print(sample_df.head())
else:
    print("[X] No data loaded")

Loading historical data for 0 symbols...
   Date range: 2022-10-19 to 2025-10-18
[OK] Successfully loaded 0/0 symbols
[X] No data loaded


In [ ]:
# Cell 5: Metric Collection Function (Ported from main.py)

def collect_stock_metrics(symbol_data, min_days=252):
    """
    Collect price, momentum, and volatility metrics for all stocks.
    
    Parameters:
    - symbol_data: dict of {symbol: DataFrame} with price history
    - min_days: minimum number of days required (default 252 trading days = 1 year)
    
    Returns:
    - DataFrame with metrics for each symbol
    """
    metrics = {}
    
    for symbol, df in symbol_data.items():
        # Need sufficient data
        if len(df) < min_days * 0.5:  # At least 50% of required days
            continue
        
        # Extract close prices
        if 'close' not in df.columns:
            continue
        
        close_prices = df['close'].values.astype(np.float64)
        
        # Handle volume if available
        if 'volume' in df.columns:
            volumes = df['volume'].values.astype(np.float64)
            avg_volume = np.mean(volumes)
        else:
            avg_volume = 0
        
        # Calculate metrics
        current_price = close_prices[-1]
        start_price = close_prices[0]
        
        # Price change over entire period
        price_change = (current_price - start_price) / start_price if start_price != 0 else 0
        
        # Momentum (21-day)
        if len(close_prices) > 21:
            prev_price = close_prices[-21]
            momentum = (current_price - prev_price) / prev_price if prev_price != 0 else 0
        else:
            momentum = 0
        
        # Volatility (standard deviation of daily returns)
        if len(close_prices) > 1:
            returns = np.diff(close_prices) / close_prices[:-1]
            returns = returns[np.isfinite(returns)]
            volatility = np.std(returns) if len(returns) > 0 else 0
        else:
            volatility = 0
        
        # Direction
        direction = 1 if price_change > 0 else -1
        
        metrics[symbol] = {
            'price': current_price,
            'price_change': price_change,
            'momentum': momentum,
            'volatility': volatility,
            'volume': avg_volume,
            'direction': direction
        }
    
    print(f"[OK] Collected metrics for {len(metrics)} stocks")
    
    if len(metrics) > 0:
        return pd.DataFrame(metrics).T
    else:
        return None


print("[OK] Metric collection function defined")

In [ ]:
# Cell 6: Clustering Function (Ported from main.py)

def perform_clustering(metrics_df):
    """
    Classify stocks into 6 Market Regimes based on Momentum and Volatility.
    
    Regimes:
    - Calm Bull: Momentum > 2%, Volatility < 1.5%
    - Volatile Bull: Momentum > 2%, Volatility > 1.5%
    - Calm Bear: Momentum < -2%, Volatility < 1.5%
    - Volatile Bear: Momentum < -2%, Volatility > 1.5%
    - Calm Sideways: -2% < Momentum < 2%, Volatility < 1.5%
    - Volatile Sideways: -2% < Momentum < 2%, Volatility > 1.5%
    
    Returns:
    - groups: dict mapping regime name to list of symbols
    - group_assignments: dict mapping symbol to regime name
    """
    if metrics_df is None or len(metrics_df) < 1:
        print(f"[!] Insufficient data for clustering")
        return None, None
    
    regime_names = [
        "Calm Bull", "Volatile Bull",
        "Calm Bear", "Volatile Bear",
        "Calm Sideways", "Volatile Sideways"
    ]
    
    groups = {name: [] for name in regime_names}
    group_assignments = {}
    
    for symbol, row in metrics_df.iterrows():
        mom = row['momentum']
        vol = row['volatility']
        
        # Determine trend
        if mom > 0.02:
            trend = "Bull"
        elif mom < -0.02:
            trend = "Bear"
        else:
            trend = "Sideways"
        
        # Determine volatility level (1.5% daily threshold)
        if vol > 0.015:
            vol_type = "Volatile"
        else:
            vol_type = "Calm"
        
        # Assign to regime
        regime = f"{vol_type} {trend}"
        groups[regime].append(symbol)
        group_assignments[symbol] = regime
    
    print(f"[OK] Clustering complete: {len(metrics_df)} stocks classified into {len(regime_names)} regimes")
    
    return groups, group_assignments


print("[OK] Clustering function defined")

In [ ]:
# Cell 7: Execute Analysis

# Collect metrics
print("Collecting stock metrics...")
metrics_df = collect_stock_metrics(symbol_data)

if metrics_df is not None:
    print(f"\n Metrics Summary:")
    print(f"   Total stocks: {len(metrics_df)}")
    print(f"\n   Descriptive Statistics:")
    print(metrics_df.describe())
    
    # Perform clustering
    print(f"\n Performing clustering...")
    groups, group_assignments = perform_clustering(metrics_df)
    
    if groups:
        # Add cluster assignments to dataframe
        metrics_df['cluster'] = metrics_df.index.map(group_assignments)
        
        print(f"\n Cluster Distribution:")
        for regime_name in sorted(groups.keys()):
            symbols = groups[regime_name]
            print(f"   {regime_name:<20}: {len(symbols):>3} stocks")
            
            if len(symbols) > 0:
                # Show average metrics for this regime
                regime_metrics = metrics_df.loc[symbols]
                avg_mom = regime_metrics['momentum'].mean()
                avg_vol = regime_metrics['volatility'].mean()
                print(f"      Avg Momentum: {avg_mom:>7.2%}  |  Avg Volatility: {avg_vol:>7.2%}")
        
        print(f"\n[OK] Analysis complete!")
    else:
        print("[X] Clustering failed")
else:
    print("[X] Metric collection failed")

In [ ]:
# Cell 8: Visualization - Scatter Plot

if metrics_df is not None and 'cluster' in metrics_df.columns:
    plt.figure(figsize=(14, 10))
    
    # Create scatter plot
    sns.scatterplot(
        data=metrics_df, 
        x='volatility', 
        y='momentum', 
        hue='cluster',
        palette='tab10',
        s=100,
        alpha=0.7,
        edgecolor='black',
        linewidth=0.5
    )
    
    # Add regime boundary lines
    plt.axhline(0.02, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Bull/Sideways threshold')
    plt.axhline(-0.02, color='blue', linestyle='--', linewidth=1.5, alpha=0.7, label='Bear/Sideways threshold')
    plt.axvline(0.015, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Calm/Volatile threshold')
    
    # Format axes as percentages
    ax = plt.gca()
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
    
    plt.title(f'Market Regime Clustering: Momentum vs Volatility\n({len(metrics_df)} stocks, analyzed on {END_DATE.date()})', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Volatility (Std Dev of Daily Returns)', fontsize=12)
    plt.ylabel('21-Day Momentum', fontsize=12)
    plt.legend(title='Market Regime', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    plt.grid(True, linestyle=':', alpha=0.4)
    plt.tight_layout()
    plt.show()
    
    print("[OK] Scatter plot generated")
else:
    print("[X] No data available for visualization")

In [ ]:
# Cell 9: Visualization - Distribution Plots

if metrics_df is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Momentum distribution
    axes[0, 0].hist(metrics_df['momentum'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(0.02, color='red', linestyle='--', linewidth=2, label='Bull threshold')
    axes[0, 0].axvline(-0.02, color='blue', linestyle='--', linewidth=2, label='Bear threshold')
    axes[0, 0].set_xlabel('Momentum', fontsize=11)
    axes[0, 0].set_ylabel('Frequency', fontsize=11)
    axes[0, 0].set_title('Momentum Distribution', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Volatility distribution
    axes[0, 1].hist(metrics_df['volatility'], bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
    axes[0, 1].axvline(0.015, color='green', linestyle='--', linewidth=2, label='Calm/Volatile threshold')
    axes[0, 1].set_xlabel('Volatility', fontsize=11)
    axes[0, 1].set_ylabel('Frequency', fontsize=11)
    axes[0, 1].set_title('Volatility Distribution', fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Price change distribution
    axes[1, 0].hist(metrics_df['price_change'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
    axes[1, 0].axvline(0, color='black', linestyle='-', linewidth=1)
    axes[1, 0].set_xlabel('Price Change (3-year)', fontsize=11)
    axes[1, 0].set_ylabel('Frequency', fontsize=11)
    axes[1, 0].set_title('Price Change Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Cluster size bar chart
    if 'cluster' in metrics_df.columns:
        cluster_counts = metrics_df['cluster'].value_counts().sort_index()
        axes[1, 1].bar(range(len(cluster_counts)), cluster_counts.values, 
                       color='mediumpurple', edgecolor='black', alpha=0.7)
        axes[1, 1].set_xticks(range(len(cluster_counts)))
        axes[1, 1].set_xticklabels(cluster_counts.index, rotation=45, ha='right', fontsize=9)
        axes[1, 1].set_xlabel('Market Regime', fontsize=11)
        axes[1, 1].set_ylabel('Number of Stocks', fontsize=11)
        axes[1, 1].set_title('Stocks per Regime', fontsize=12, fontweight='bold')
        axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("[OK] Distribution plots generated")
else:
    print("[X] No data available for visualization")

In [ ]:
# Cell 10: Regime Details Table

if metrics_df is not None and groups is not None:
    print("=" * 100)
    print("MARKET REGIME ANALYSIS")
    print("=" * 100)
    
    for regime_name in sorted(groups.keys()):
        symbols_list = groups[regime_name]
        
        if len(symbols_list) == 0:
            continue
        
        print(f"\n{'─' * 100}")
        print(f"{'REGIME: ' + regime_name:<50} | Members: {len(symbols_list)}")
        print(f"{'─' * 100}")
        
        # Get metrics for this regime
        regime_df = metrics_df.loc[symbols_list]
        
        # Summary statistics
        print(f"\n  Summary Statistics:")
        print(f"    Avg Momentum:     {regime_df['momentum'].mean():>8.2%}  (min: {regime_df['momentum'].min():>7.2%}, max: {regime_df['momentum'].max():>7.2%})")
        print(f"    Avg Volatility:   {regime_df['volatility'].mean():>8.2%}  (min: {regime_df['volatility'].min():>7.2%}, max: {regime_df['volatility'].max():>7.2%})")
        print(f"    Avg Price Change: {regime_df['price_change'].mean():>8.2%}  (3-year)")
        print(f"    Avg Price:        ${regime_df['price'].mean():>8.2f}")
        
        # List members
        print(f"\n  Members:")
        sorted_members = sorted(symbols_list)
        for i in range(0, len(sorted_members), 10):
            row = sorted_members[i:i+10]
            print(f"    {', '.join(row)}")
    
    print(f"\n{'=' * 100}\n")
else:
    print("[X] No regime data available")

In [ ]:
# Cell 11: Top Performers by Regime

if metrics_df is not None and 'cluster' in metrics_df.columns:
    print("=" * 100)
    print("TOP 5 PERFORMERS BY REGIME (by 21-day momentum)")
    print("=" * 100)
    
    for regime_name in sorted(groups.keys()):
        symbols_list = groups[regime_name]
        
        if len(symbols_list) == 0:
            continue
        
        regime_df = metrics_df.loc[symbols_list]
        top_performers = regime_df.nlargest(5, 'momentum')
        
        print(f"\n{regime_name}:")
        print(f"{'  Rank':<7} {'Symbol':<8} {'Momentum':<12} {'Volatility':<12} {'Price':<10}")
        print(f"  {'-'*60}")
        
        for idx, (symbol, row) in enumerate(top_performers.iterrows(), 1):
            print(f"  {idx:<7} {symbol:<8} {row['momentum']:>9.2%}   {row['volatility']:>9.2%}   ${row['price']:>8.2f}")
    
    print(f"\n{'=' * 100}\n")
else:
    print("[X] No clustering data available")

In [ ]:
# Cell 12: Export Results (Optional)

if metrics_df is not None:
    # Save metrics to CSV
    output_file = Path("metrics_output.csv")
    metrics_df.to_csv(output_file)
    print(f"[OK] Metrics exported to: {output_file.absolute()}")
    
    # Save regime assignments
    if 'cluster' in metrics_df.columns:
        regime_file = Path("regime_assignments.csv")
        regime_export = metrics_df[['cluster', 'momentum', 'volatility', 'price', 'price_change']]
        regime_export.to_csv(regime_file)
        print(f"[OK] Regime assignments exported to: {regime_file.absolute()}")
else:
    print("[X] No data to export")